### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [13]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from pathlib import Path

In [10]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error: {e}")
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")



Found 2 PDF files to process
Processing: The Art of Running Faster PDF.pdf
 Loaded 151 pages
Processing: book1.pdf
 Loaded 399 pages
Total documents loaded: 550


In [11]:
all_pdf_documents

[Document(metadata={'producer': 'jsPDF 2.5.2', 'creator': 'BOOKEY', 'creationdate': '2025-11-20T03:40:35+08:00', 'title': 'The Art of Running Faster', 'subject': 'Health & Sports,Sports,Personal Development,Body & Soul', 'author': 'Julian Goater', 'keywords': 'The Art of Running Faster', 'source': '../data/pdf/The Art of Running Faster PDF.pdf', 'total_pages': 151, 'page': 0, 'page_label': '1', 'source_file': 'The Art of Running Faster PDF.pdf', 'file_type': 'pdf'}, page_content='The Art of Running Faster\nPDF\nJulian Goater'),
 Document(metadata={'producer': 'jsPDF 2.5.2', 'creator': 'BOOKEY', 'creationdate': '2025-11-20T03:40:35+08:00', 'title': 'The Art of Running Faster', 'subject': 'Health & Sports,Sports,Personal Development,Body & Soul', 'author': 'Julian Goater', 'keywords': 'The Art of Running Faster', 'source': '../data/pdf/The Art of Running Faster PDF.pdf', 'total_pages': 151, 'page': 1, 'page_label': '2', 'source_file': 'The Art of Running Faster PDF.pdf', 'file_type': 'pd

In [15]:
### Text splitting get into chunks

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    return split_docs




In [16]:
chunks = split_documents(all_pdf_documents)


Split 550 documents into 1301 chunks


In [22]:
chunks

[Document(metadata={'producer': 'jsPDF 2.5.2', 'creator': 'BOOKEY', 'creationdate': '2025-11-20T03:40:35+08:00', 'title': 'The Art of Running Faster', 'subject': 'Health & Sports,Sports,Personal Development,Body & Soul', 'author': 'Julian Goater', 'keywords': 'The Art of Running Faster', 'source': '../data/pdf/The Art of Running Faster PDF.pdf', 'total_pages': 151, 'page': 0, 'page_label': '1', 'source_file': 'The Art of Running Faster PDF.pdf', 'file_type': 'pdf'}, page_content='The Art of Running Faster\nPDF\nJulian Goater'),
 Document(metadata={'producer': 'jsPDF 2.5.2', 'creator': 'BOOKEY', 'creationdate': '2025-11-20T03:40:35+08:00', 'title': 'The Art of Running Faster', 'subject': 'Health & Sports,Sports,Personal Development,Body & Soul', 'author': 'Julian Goater', 'keywords': 'The Art of Running Faster', 'source': '../data/pdf/The Art of Running Faster PDF.pdf', 'total_pages': 151, 'page': 1, 'page_label': '2', 'source_file': 'The Art of Running Faster PDF.pdf', 'file_type': 'pd

### Embedding and VectorStoreDB

In [18]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name:str = "sentence-transformers/all-MiniLM-L6-v2"):
        """
            Initialize the embedding manager

            Args:
                model_name
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loades successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str] ) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()


## initialize the embedding manager

em = EmbeddingManager()
em

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4427.14it/s]


Model loades successfully. Embedding dimension: 384


/tmp/ipykernel_1179227/2593061190.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loades successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [69]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name:str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    

    def _initialize_store(self):
        try:
            
            # Create persistent ChromaDB client
            path_dir = Path(self.persist_directory).resolve()
            os.makedirs(path_dir, exist_ok = True)
            self.client = chromadb.PersistentClient(path=path_dir)

            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        for index, (doc, embed) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{index}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = index
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embed.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Failes to add to the collection {e}")
            raise


vectorstore = VectorStore()



Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [ ]:
texts = [chunk.page_content for chunk in chunks]
embeddings = em.generate_embeddings(texts)


Generating embeddings for 1301 texts...


Batches: 100%|██████████| 41/41 [00:50<00:00,  1.23s/it]

Generated embeddings with shape: (1301, 384)


In [70]:
vectorstore.add_documents(chunks, embeddings)

Adding 1301 documents to vector store...
Successfully added 1301 documents to vector store
Total documents in collection: 1301


## Retriever Pipeline from VectorStore

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str,Any]]:
        """
            Retrieve relevant documents for a query

            Args:
                query: The search query
                top_k: Number of top results to return
                score_threshold: Minimum similarity score threshold

            Returns:
                List of dictionaries containing retrieved documents and metadata
        """
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                'id': doc_id,
                                'content': document,
                                'metadata': metadata,
                                'similarity_score': similarity_score,
                                'distance': distance,
                                'rank' : i + 1

                            }
                        )

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

        except Exception as e:
            print(f"Problem searching in vector store: {e}")
            return []
        
        return retrieved_docs

retriever = RAGRetriever(vectorstore, em)
retriever


In [74]:
retriever.retrieve(
    "How can you run faster without getting so tired and with the heart so heartbumbed?"
)

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.75it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_c916cafb_59',
  'content': 'Best Quotes from The Art of Running\nFaster by Julian Goater with Page\nNumbers\nView on Bookey Website and Generate Beautiful Quote Images\nChapter 1 | Quotes From Pages 44-75\n1.If you want to go faster, you’ve got to extend those\nlimits. Otherwise, they’re going to stay right where\nthey are, keeping you from going faster than you\ngo now.\n2.You should get out of breath—at least some of the time—in\nmost of your training sessions.\n3.Your training should address all of these factors:\nImprovements in your speed, skill, and suppleness will also\nhelp improve your stamina.\n4.Let’s take these misconceptions one by one. A heart-rate\nmonitor has some value, to be sure, but it’s only a tool, not\nthe be-all and end-all of training.\n5.But many runners’ training is one-dimensional.\n6.Remember that you can go faster only by increasing your',
  'metadata': {'page': 59,
   'creationdate': '2025-11-20T03:40:35+08:00',
   'total_pages': 151,
   'pag

## Integration VectorDB Context pipeline With LLM outpu

In [91]:
### Simple RAG pipeline with Claude LLM
from langchain_anthropic import ChatAnthropic
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatAnthropic(
    model = "claude-haiku-4-5",
    temperature = 0.5,
    #max_tokens = 120
)

llm


ChatAnthropic(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0', 'langchain-anthropic': '1.7.2'}}, profile={'name': 'Claude Haiku 4.5 (latest)', 'release_date': '2025-10-15', 'last_updated': '2025-10-15', 'open_weights': False, 'max_input_tokens': 200000, 'max_output_tokens': 64000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_call_streaming': True}, model='claude-haiku-4-5', max_tokens=64000, temperature=0.5, anthropic_api_url='https://api.anthropic.com', anthropic_api_key=SecretStr('**********'), model_kwargs={})

In [90]:
# Simple RAG function: retrieve context + generate response

def rag_qa(query: str, retriever: RAGRetriever, llm: ChatAnthropic, top_k: int = 5) -> str:

    context = retriever.retrieve(
        query = query,
        top_k = top_k
    )
    context_content = "\n\n".join([c['content'] for c in context]) if context else ""
    if not context:
        return "No relevant context found to answer the question"
    
    ## Generate the answer using claude
    system_prompt = f""" Use the following context to answer the question concisely.

    Context: 
    {context_content}
    """

    messages = [
        {
            "role" : "system", "content": system_prompt
        },
        {
            "role" : "human", "content": query
        }
    ]
    
    response = llm.invoke(messages)

    return response



response = rag_qa("How can you run faster without getting so tired and with the heart so heartbumbed?", retriever, llm)

print(response.content)



Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 84.94it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


# Running Faster Without Excessive Fatigue

Based on *The Art of Running Faster*, the key is **not to avoid getting tired**, but rather to train intelligently:

## The Counterintuitive Truth

The book actually argues against your goal. You **should** get out of breath during training because:

- Getting out of breath improves your **lung capacity**
- It strengthens your **heart** (speedwork makes it bigger and stronger)
- It builds **stamina** for faster running

## Smart Training Approach

Rather than avoiding fatigue entirely, the book recommends:

1. **Varied Training** - Don't always run at the same pace. Mix speeds, surfaces, and gradients to build resilience and avoid overuse injuries

2. **Balanced Program** - Include:
   - Speed work (gets you out of breath)
   - Strength training
   - Skill development
   - Suppleness work

3. **Gauge Effort by Breathing** - Don't rely solely on heart rate monitors; listen to your body's signals

4. **Recovery Matters** - "The primary aim of e

## Enhanced RAG Pipeline Features

In [95]:
# Advanced RAG Pipeline: Streaming, Citations, History, Summarization

from typing import List, Dict, Any
import time


class AdvancedRAGPipeline:
    def __init__(self, retriever: RAGRetriever, llm: ChatAnthropic):
        self.retriever = retriever
        self.llm = llm
        self.history = [] # Save query history
    
    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict:
        results = self.retriever.retrieve(question, top_k = top_k, score_threshold = min_score)
        if not results:
            print("No context obtained from RAG")
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc["content"] for doc in results])
            sources = [{
                "source" : doc["metadata"].get("source", "unknown"),
                "title" : doc["metadata"].get("title", "unknown"),
                "content_length": doc["metadata"].get("content_length", "unknown"),
                "page_number": f"{doc["metadata"].get("page", "unknown")}/{doc["metadata"].get("total_pages", "unknown")}",
                "score": doc["similarity_score"]
            } for doc in results]

            # Streaming answer simulation
            prompt = f"""
                Use the following context to answer the quesion concisely.

                Contest:
                {context}

                Question:
                {question}
            """

            if stream:
                print("Streaming anser:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i: i+80], end = "", flush = True)
                    time.sleep(0.05)
                print()
            
            response = self.llm.invoke([prompt.format(context = context, question = question)])
            answer = response.content

            # Add citations to answer
            citations = [
                f"({i+1}) File: {source.get("title")} with source {source.get("source")}. Pag {source.get("page")}/{source.get("total_pages")}"
                for i, source in enumerate(sources)
            ]
            answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

            summary = None
            if summarize and answer:
                summary_prompt = f"Summarize the following answer in 2 sentences: \n{answer}"
                summary_resp = self.llm.invoke([summary_prompt])
                summary = summary_resp.content
            
            # Store query history
            self.history.append({
                "question": question,
                "answer": answer,
                "summary": summary,
                "sources" : sources
            })

            return {
                "question": question,
                "answer": answer,
                "summary": summary,
                "sources" : sources,
                "history" : self.history
            }


adv_rag = AdvancedRAGPipeline(retriever, llm)


In [96]:
result = adv_rag.query("What is the best way to prepare a marathon?", top_k = 3, min_score = 0.1, stream = True, summarize = True)
print(f"Final answer:\n {result["answer"]}")
print(f"Summary: {result["summary"]}")
print(f"Sources: {result["sources"]}")

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 48.49it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming anser:

                Use the following context to answer the quesion concisely.

                Contest:
                preparation during the weeks prior to the big race are crucial.
3.Question
What are some key strategies for tapering before a race?
Answer:Key strategies for tapering before a race include
reducing trai

ning intensity while maintaining the frequency
of sessions, integrating easy runs with some short, fast efforts
to keep the body feeling quick and responsive, and ensuring
proper hydration without overindulging.
4.Question
Why is mental preparation important in the lead-up to a
race?
Answer:Mental preparation shapes expectations and can lead
to performances beyond one's usual capabilities. By setting
positive expectations and visualizing success, athletes can
better access their full potential when it counts.
5.Question
What should athletes focus on during the days leading up
to a race?
Answer:During the days before a race, athletes should
prioritize how their body feels rather than chasing additional

: Engage in relaxed, unstructured activities to recover
physically and mentally after a competitive season.
- 
Basic Conditioning
: Improve overall strength and resilience through strength
training and gradual mileage increases without high intensity.
- 
Endurance Base
: Build a solid fo